In [18]:
%pip install pandas
%pip install python-calamine

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import pandas as pd 
import numpy as np
import re
import openpyxl

In [20]:
data = pd.read_excel(r'C:\Users\KS\Desktop\K2\K2 รายงานสต็อกการ์ด พร้อมทุน.xls' ,engine='calamine',header=11,usecols="A:F",dtype={'Unnamed: 3': str,'ลด ': str,'คงเหลือ ': str})
data.rename(columns={'Unnamed: 0':'DATE','Unnamed: 1':'Bill','เพิ่ม ':'details','Unnamed: 3':'value','ลด ':'sale','คงเหลือ ':'balance'} ,inplace=True)

# DATE == รหัสสินค้า มีชื่อสินค้าอยู่ ถ้าเป็นค่าว่างให้ดึงชื่อจากแถวก่อนหน้า
data['product_id'] = data.loc[data['DATE'] =='รหัสสินค้า', 'sale']
data['product_id'] = data['product_id'].ffill()

# DATE == คลัง มีชื่อสินค้าอยู่ ถ้าเป็นค่าว่างให้ดึงชื่อจากแถวก่อนหน้า
data['unit'] = data.loc[data['DATE'].astype(str).str.strip() == 'คลัง', 'balance']
data['unit'] = data['unit'].ffill()
data['unit'] = data['unit'].str.extract(r'(\d+)').fillna(0).astype(int)

# แปลงตรงๆ โดยบอกสไตล์ปฏิทินสากลไปก่อน
data['DATE'] = pd.to_datetime(data['DATE'], format='%d/%m/%Y', errors='coerce')

# ลบปีออก 543 ปี (ใช้ DateOffset)
data['DATE'] = data['DATE'] - pd.DateOffset(years=543)
data['DATE'] = pd.to_datetime(data['DATE'])
data.dropna(subset=['DATE'], inplace=True)

In [21]:

def parse_pack_piece(series_val, series_unit):
    """ฟังก์ชันแยกจำนวนแพ็ก (front) และเศษชิ้นย่อย (back) อัตโนมัติ โดยใช้ Logic

    len(str(unit - 1)) กำหนดทศนิยมรายบรรทัด
    """

    def format_by_unit(v, u):
        try:
            val_float = float(v)
            unit_int = int(float(u))

            if unit_int <= 1:
                return f'{val_float:.1f}'

            decimals = len(str(unit_int - 1))
            return f'{val_float:.{decimals}f}'
        except:
            return '0.0'

    # 1. แปลงค่าโดยใช้ .index จาก series_val เดิม เพื่อป้องกัน Index Mismatch
    s_clean = pd.Series(
        [format_by_unit(v, u) for v, u in zip(series_val, series_unit)],
        index=series_val.index,  # <-- ล็อก Index ให้ตรงกับ DataFrame ต้นทาง
    )

    # 2. แยกด้วย .str.split('.') ชัวร์และเร็วกว่า Regex
    split_df = s_clean.str.split('.', expand=True)

    # 3. ดึง front (หน้าจุด)
    front = pd.to_numeric(split_df[0], errors='coerce').fillna(0).astype(int)

    # 4. ดึง back (หลังจุด)
    back = pd.to_numeric(split_df[1], errors='coerce').fillna(0).astype(int)

    return front, back


# ==========================================
# 🚀 โค้ดส่วนการประมวลผลข้อมูล
# ==========================================

# แปลง unit เป็นตัวเลขแท้ๆ ป้องกันคูณแล้วพัง
unit_num = pd.to_numeric(data['unit'], errors='coerce').fillna(1).astype(int)

# 1. แยกหน่วย Import (Value)
data['front_value'], data['back_value'] = parse_pack_piece(
    data['value'], data['unit']
)

# 2. แยกหน่วย Export (Sale)
data['front_sale'], data['back_sale'] = parse_pack_piece(
    data['sale'], data['unit']
)

# 3. แยกหน่วย Balances
data['front_balance'], data['back_balance'] = parse_pack_piece(
    data['balance'], data['unit']
)

# 4. คำนวณข้อย่อยรวม (ใช้ unit_num ที่เป็นตัวเลขแล้ว)
data['import'] = data['back_value'] + (unit_num * data['front_value'])
data['export'] = data['back_sale'] + (unit_num * data['front_sale'])
data['balances'] = data['back_balance'] + (unit_num * data['front_balance'])

## Hybrid Robust Z-score (Global + Rolling) พร้อมคำนวณส่วนต่างและช่วงเวลาบิลหาย

**สิ่งที่เพิ่มเข้ามาใน V3:**
- **คอลัมน์ตรวจสอบบิลย้อนหลัง (Actionable Columns):** สำหรับเคส `🔴 บิลหายทั้งใบ` ระบบจะระบุวันและช่วงวันที่พนักงานบัญชี/คลังสินค้าต้องไปตามหาเอกสารทันที:
  1. `suspect_start_date` (วันที่เริ่มน่าสงสัย) = วันที่ของเข้าล่าสุด + 1 วัน
  2. `suspect_end_date` (วันที่สิ้นสุดการสงสัย) = วันที่ทำรายการเบิกออก/ขายปัจจุบัน
  3. `suspect_date_range` (ช่วงเวลาที่ต้องไปย้อนดูบิล) = รวมเป็นข้อความให้อ่านง่าย เช่น `2026-07-01 ถึง 2026-08-05`

In [22]:
#%pip install scipy
#%pip install openpyxl
#%pip install scikit-learn

In [29]:
import math
import numpy as np
import pandas as pd
from scipy import stats
from rapidfuzz import fuzz, process

print("--- 🚀 Engine Started: Full Dynamic 5-Rules Audit & Physical Stock Reconciliation ---")

# ==========================================
# 1. เตรียม Data & Base Cleaning (ส่วนที่ 1)
# ==========================================
df = data[
    [
        'DATE',
        'Bill',
        'details',
        'product_id',
        'import',
        'export',
        'balances',
    ]
].copy()

df.columns = df.columns.str.strip()
df['product_id'] = df['product_id'].astype(str).str.strip()
df['details'] = df['details'].astype(str).str.strip()
df['Bill'] = df['Bill'].astype(str).str.strip()
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values(by=['product_id', 'DATE']).reset_index(drop=True)

max_date_in_dataset = df['DATE'].max()

# 🎯 Dynamic Unit: ตรวจจับบิล Supplier (IB, IBK, DM) และหา unit เล็กสุดรายสินค้า
trade_prefix_pattern = r'^(?:IB|IBK|DM)'
df['bill_code_clean'] = df['Bill'].str.upper()
df['details_clean'] = df['details'].str.upper()

df['is_supplier_trade'] = (
    df['bill_code_clean'].str.contains(trade_prefix_pattern, na=False) | 
    df['details_clean'].str.contains(trade_prefix_pattern, na=False)
)

sku_unit_map = df[(df['import'] > 0) & (df['is_supplier_trade'])].groupby('product_id')['import'].min().to_dict()
if not sku_unit_map:
    sku_unit_map = df[df['import'] > 0].groupby('product_id')['import'].min().to_dict()

df['unit'] = df['product_id'].map(sku_unit_map).fillna(1.0)

# กรองบิลรับคืน / NV / ปรับปรุง
exclude_keywords = ['รับคืน', 'NV', 'ปรับปรุง', 'ยกมา', 'ยอดยกมา', 'adjust']
pattern_exclude = '|'.join(exclude_keywords)
df['is_return_bill'] = np.where(
    df['is_supplier_trade'],
    False,
    df['details_clean'].str.contains(pattern_exclude, na=False)
)

df['net_export'] = np.where(df['is_return_bill'], -df['import'], df['export'])

# ==========================================
# 2. Dynamic Benchmark Selection (MODE / MEDIAN / IQR)
# ==========================================
import_history = df[(df['import'] > 0) & (~df['is_return_bill'])].copy()

def calculate_adaptive_stats(df_imp):
    if df_imp.empty:
        return pd.DataFrame()

    stats_list = []
    for pid, group in df_imp.groupby('product_id'):
        imp_vals = group['import'].values
        
        # 1) MODE
        m = stats.mode(imp_vals, keepdims=False)
        mode_val = m.mode if np.isscalar(m.mode) else m.mode[0]
        mode_freq = np.mean(imp_vals == mode_val)
        
        # 2) MEDIAN & IQR
        median_val = np.median(imp_vals)
        q1, q3 = np.percentile(imp_vals, 25), np.percentile(imp_vals, 75)
        iqr_val = q3 - q1
        
        # 3) Dynamic Decision Logic รายสินค้า
        if mode_freq >= 0.20:
            chosen_method = 'MODE'
            selected_size = mode_val
            spread = np.mean(np.abs(imp_vals - mode_val))
        elif iqr_val > 0 and (iqr_val / (median_val + 1e-5)) > 0.5:
            chosen_method = 'IQR'
            selected_size = median_val
            spread = iqr_val
        else:
            chosen_method = 'MEDIAN'
            selected_size = median_val
            spread = np.median(np.abs(imp_vals - median_val))
            
        stats_list.append({
            'product_id': pid,
            'chosen_method': chosen_method,
            'selected_import_size': selected_size,
            'spread_val': spread if spread > 0 else 1.0
        })
        
    return pd.DataFrame(stats_list)

if not import_history.empty:
    stats_df = calculate_adaptive_stats(import_history)
    df = df.merge(stats_df, on='product_id', how='left')

df['selected_import_size'] = df['selected_import_size'].fillna(df['import'])
df['spread_val'] = df['spread_val'].fillna(1.0)
df['zscore_stat'] = (df['import'] - df['selected_import_size']) / df['spread_val']

# ==========================================
# 3. คำนวณประเมินผล 5 สมมติฐานหลัก (5 Core Rules)
# ==========================================

# --- Rule 1: Ghost Overstock ---
sku_avg_sales = df[df['export'] > 0].groupby('product_id')['export'].mean().reset_index().rename(columns={'export': 'avg_sales_qty'})
df = df.merge(sku_avg_sales, on='product_id', how='left')
df['avg_sales_qty'] = df['avg_sales_qty'].fillna(df['selected_import_size'])

df['is_ghost_overstock'] = (
    (df['balances'] > (1.5 * df['avg_sales_qty'])) & 
    (df['import'] > 0) & 
    (~df['is_return_bill']) & 
    (df['export'] == 0)
)

# --- Rule 2: Missing Inbound (Negative Balance) ---
df['is_negative'] = df['balances'] < 0
df['is_first_negative'] = (df['is_negative'] == True) & (df.groupby('product_id')['is_negative'].shift(1, fill_value=False) == False)

sku_max_deficit = df[df['balances'] < 0].groupby('product_id')['balances'].min().abs().to_dict()
df['max_deficit'] = df['product_id'].map(sku_max_deficit).fillna(0)

def snap_to_pack_size(deficit, pack_size):
    if pack_size <= 0: return deficit
    return math.ceil(deficit / pack_size) * pack_size if deficit >= pack_size else pack_size

# --- Rule 3: Bar Swap / Unit Swap (Dynamic unit) ---
df['alt_import_swapped'] = np.where(
    (df['import'] > 0) & (df['import'] < df['unit']),
    df['import'] * df['unit'],
    df['import']
)

df['rolling_7d_export'] = df.groupby('product_id')['export'].transform(lambda x: x.rolling(7, min_periods=1).sum())

df['is_bar_swap'] = (
    (df['import'] > 0) & 
    (~df['is_return_bill']) & 
    (df['import'] < df['unit']) & 
    (
        (df['max_deficit'] > 0) | 
        (df['rolling_7d_export'] > df['selected_import_size']) |
        (df['alt_import_swapped'] >= df['unit'])
    )
)

# --- Rule 4: Outlier Detection ---
df['is_outlier_import'] = (
    (~df['is_bar_swap']) & 
    (df['import'] > 0) & 
    (~df['is_return_bill']) & 
    (df['selected_import_size'] > 0) & 
    ((df['import'] >= (df['selected_import_size'] * 2.5)) | (df['zscore_stat'] >= 2.5))
)

# --- Rule 5: Data Gap & Missing Date Range ---
df['prev_doc_date'] = df.groupby('product_id')['DATE'].shift(1)
df['days_since_last_doc'] = (df['DATE'] - df['prev_doc_date']).dt.days.fillna(0)
sku_median_gap = df.groupby('product_id')['days_since_last_doc'].median().to_dict()
df['normal_sku_gap'] = df['product_id'].map(sku_median_gap).fillna(3.0)

df['is_data_gap'] = df['days_since_last_doc'] > (df['normal_sku_gap'] * 3.0)
df['Missing_Date_Range'] = np.where(
    df['is_data_gap'],
    df['prev_doc_date'].dt.strftime('%d/%m/%Y') + ' ถึง ' + df['DATE'].dt.strftime('%d/%m/%Y'),
    '-'
)

# ==========================================
# 4. Reconstruction Engine & Expected Calculations
# ==========================================
def calculate_reconciled_import(row):
    if row['is_bar_swap']:
        return row['alt_import_swapped']
    if row['is_ghost_overstock']:
        return 0.0
    if row['is_first_negative'] and row['max_deficit'] > 0:
        return snap_to_pack_size(row['max_deficit'], row['selected_import_size'])
    if row['is_outlier_import']:
        return row['selected_import_size']
    return row['import']

df['Expected_Import'] = df.apply(calculate_reconciled_import, axis=1)

df['net_flow'] = df['import'] - df['export']
df['adjusted_net_flow'] = df['Expected_Import'] - df['net_export']
first_balance_actual = df.groupby('product_id')['balances'].transform('first')
first_net_flow_actual = df.groupby('product_id')['net_flow'].transform('first')
true_initial_balance = first_balance_actual - first_net_flow_actual
df['True_Reconciled_Stock'] = true_initial_balance + df.groupby('product_id')['adjusted_net_flow'].cumsum()

# ==========================================
# 5. Labeling (ส่วนที่ 1)
# ==========================================
def get_anomaly_score(row):
    if row['is_bar_swap']: return 98
    if row['is_ghost_overstock']: return 95
    if row['is_first_negative']: return 90
    if row['is_outlier_import']: return 80
    if row['is_data_gap']: return 65
    return 0

def get_anatomy_label(row):
    if row['is_bar_swap']:
        return f"⚠️ Rule 3: คีย์สลับหน่วย (คีย์ {int(row['import'])} -> ควรเป็น {int(row['alt_import_swapped'])})"
    if row['is_ghost_overstock']:
        return f"⚠️ Rule 1: สต็อกค้างเกินปกติ ({int(row['balances'])} ชิ้น > 1.5x Avg Sales) -> ปรับเป็น 0"
    if row['is_first_negative']:
        return f"⚠️ Rule 2: สต็อกติดลบจากการลืมคีย์รับเข้า (ขาด {int(row['max_deficit'])} ชิ้น)"
    if row['is_outlier_import']:
        return f"⚠️ Rule 4: เอกสารคีย์ยอดโดดผิดปกติ (คีย์ {int(row['import'])} -> ควรเป็น {int(row['selected_import_size'])})"
    if row['is_data_gap']:
        return f"⚠️ Rule 5: เอกสารตกหล่นช่วง {row['Missing_Date_Range']} (เว้น {int(row['days_since_last_doc'])} วัน)"
    return "⚪ ปกติ"

df['Anomaly_Score'] = df.apply(get_anomaly_score, axis=1)
df['Anatomy_Label'] = df.apply(get_anatomy_label, axis=1)

output_cols = [
    'DATE', 'product_id', 'Bill', 'details', 'unit', 'import', 'export', 
    'balances', 'Expected_Import', 'True_Reconciled_Stock', 'Missing_Date_Range', 
    'Anomaly_Score', 'Anatomy_Label'
]
df_master_reconciled = df[output_cols].copy()


# ==========================================
# 6. ฟังก์ชัน Module 2: Reconciliation With File 2 (Physical Count) - Complete Robust Version
# ==========================================
def run_module2_reconciliation(df_reconciled, file2_path_or_df):
    if isinstance(file2_path_or_df, str):
        file2_df = pd.read_excel(file2_path_or_df)
    else:
        file2_df = file2_path_or_df.copy()
        
    file2_df.columns = file2_df.columns.str.strip()
    
    # 1. ตรวจหาคอลัมน์ชื่อสินค้า และ ยอดนับจริง Dynamic
    detail_col = [c for c in file2_df.columns if any(k in c.lower() for k in ['detail', 'item', 'name', 'สินค้า', 'รายการ'])]
    detail_col = detail_col[0] if detail_col else file2_df.columns[0]
    
    stock_col = [c for c in file2_df.columns if any(k in c.lower() for k in ['stock', 'actual', 'qty', 'นับ', 'จริง', 'ยอด'])]
    stock_col = stock_col[0] if stock_col else file2_df.columns[1]
    
    print(f"🔍 อ่านไฟล์ 2 สำเร็จ: คอลัมน์สินค้า='{detail_col}', คอลัมน์สต็อกจริง='{stock_col}'")

    # Clean ข้อความและแปลงเป็น String ทั้งหมด ป้องกัน Type Mismatch Error
    file2_df[detail_col] = file2_df[detail_col].fillna('').astype(str).str.strip()
    df_reconciled['details_clean'] = df_reconciled['details'].fillna('').astype(str).str.strip()
    
    physical_items = [x for x in file2_df[detail_col].unique() if x != '']
    master_items = [x for x in df_reconciled['details_clean'].unique() if x != '']
    
    # 2. Matching Engine (Fuzzy + Fallback)
    matched_mapping = {}
    for name in master_items:
        # ลอง Fuzzy Match
        match = process.extractOne(name, physical_items, scorer=fuzz.WRatio, score_cutoff=30)
        if match:
            matched_mapping[name] = match[0]
        else:
            # ถ้า Fuzzy ไม่เจอ ลองค้นแบบ Partial Match (คำบางส่วน)
            for p_item in physical_items:
                if (p_item in name) or (name in p_item):
                    matched_mapping[name] = p_item
                    break
            
    df_reconciled['matched_physical_name'] = df_reconciled['details_clean'].map(matched_mapping)
    df_reconciled['matched_physical_name'] = df_reconciled['matched_physical_name'].fillna(df_reconciled['details_clean'])
    
    # เตรียมข้อมูลฝั่ง Physical Count
    file2_clean = file2_df[[detail_col, stock_col]].drop_duplicates(subset=[detail_col]).copy()
    file2_clean.columns = ['matched_physical_name', 'actual_stock']
    
    # 🎯 3. MERGE แบบ LEFT JOIN (ไม่ตัดรายการหลักทิ้ง แม้จะหาคู่ในไฟล์ที่ 2 ไม่เจอ)
    m2_df = df_reconciled.merge(file2_clean, on='matched_physical_name', how='left')
    
    # 4. คำนวณ Variance
    m2_df['actual_stock'] = pd.to_numeric(m2_df['actual_stock'], errors='coerce')
    
    # คำนวณ Stock Variance (กรณีไม่มีข้อมูลในไฟล์ 2 จะขึ้นเป็น NaN)
    m2_df['Stock_Variance'] = np.where(m2_df['actual_stock'].isna(), np.nan, m2_df['balances'] - m2_df['actual_stock'])
    
    m2_df['Variance_Analysis'] = np.where(
        m2_df['actual_stock'].isna(),
        '❓ ไม่พบข้อมูลในไฟล์นับจริง',
        np.where(
            m2_df['Stock_Variance'] > 0,
            '➕ เกินจริงในระบบ (ติดบวก)',
            np.where(
                m2_df['Stock_Variance'] < 0,
                '➖ ขาดจากระบบ (ติดลบ)',
                '✅ ยอดตรงกัน'
            )
        )
    )
    
    matched_count = m2_df['actual_stock'].notna().sum()
    print(f"✅ ประมวลผลสำเร็จ! ข้อมูลทั้งหมด {len(m2_df)} รายการ (จับคู่ตรงกับไฟล์ที่ 2 ได้ {matched_count} รายการ)")
    return m2_df


--- 🚀 Engine Started: Full Dynamic 5-Rules Audit & Physical Stock Reconciliation ---


In [28]:
# ==========================================
# 7. Execution: สั่งรัน ประมวลผล และ Export
# ==========================================

# 📌 7.1 รันส่วนที่ 2 กับไฟล์ K2.xlsx บน Desktop
df_module2_report = run_module2_reconciliation(df_master_reconciled, r'C:\Users\KS\Desktop\K2.xlsx')

# 📌 7.2 โชว์ตารางตัวอย่างบนจอภาพ
print("\n--- 📊 ตัวอย่างตารางผลลัพธ์ Module 2 (10 รายการแรก) ---")
print(df_module2_report[['DATE', 'product_id', 'details', 'balances', 'actual_stock', 'Stock_Variance', 'Variance_Analysis', 'Anatomy_Label']].head(10).to_string(index=False))

# 📌 7.3 Export ออกเป็น Excel
export_path = r'C:\Users\KS\Desktop\df_module2_report.xlsx'
df_module2_report.to_excel(export_path, index=False, engine='openpyxl')
print(f"\n🎉 บันทึกไฟล์เรียบร้อยแล้วที่: {export_path}")

🔍 อ่านไฟล์ 2 สำเร็จ: คอลัมน์สินค้า='date', คอลัมน์สต็อกจริง='actual_stock'
⚠️ ไม่พบคู่ Fuzzy Match, สลับไปใช้ Direct Matching...
✅ ประมวลผลสำเร็จ! พบรายการที่ตรงกันทั้งหมด 0 บรรทัดรายการ

--- 📊 ตัวอย่างตารางผลลัพธ์ Module 2 (10 รายการแรก) ---
Empty DataFrame
Columns: [DATE, product_id, details, balances, actual_stock, Stock_Variance, Variance_Analysis, Anatomy_Label]
Index: []

🎉 บันทึกไฟล์เรียบร้อยแล้วที่: C:\Users\KS\Desktop\df_module2_report.xlsx


In [25]:
df_module2_report.to_excel( r'C:\Users\KS\Desktop\df_module2_report.xlsx',index=False,engine='openpyxl')